# Understand the saved results

## Objective

Read saved forecasts and scores, then export them for analysis.

Run notebook 4 first. Reading results does not train models or calculate replacement scores. A saved run contains predictions, scores, model files, settings and source records.

MAE of actuals `[0, 1, 2]` and predictions `[-2, 1, 3]` is `1`. Negative forecasts stay negative. Missing actuals follow your study's rule; an unavailable score is `None`, not zero.


## Step 1: Choose a place for the example files

We will keep data and results outside the library folder. The printed path shows where to find them. Set the environment variable only if you want a different folder.


In [ ]:
# Import the tools used in this step.
import os
from pathlib import Path

# Choose the workspace without changing the library checkout.
work = (
    Path(os.environ.get("AUTOFORECAST_TUTORIAL_DIR", Path.home() / "autoforecast-tutorials"))
    .expanduser()
    .resolve()
)
# Create the folder if this is your first visit.
work.mkdir(parents=True, exist_ok=True)
print(work)

### Understand the output

You should see the full workspace path. Open that folder to find the files created in these lessons. This path is a folder, not a forecast result.


## Step 2: Open the saved run

We will use the path saved by notebook 4. Loading checks its files; it does not train the model again.


In [ ]:
# Import the tools used in this step.
from autoforecast.study_workflow import load_forecast_workflow

# Check and open the saved run without fitting again.
result = load_forecast_workflow((work / "latest-run.txt").read_text().strip())

### Understand the output

No output is expected. `result` now refers to the checked saved run. If files have been moved, removed or changed, loading may fail; keep the linked study and tuning folders with the workflow.


## Step 3: Inspect scores and predictions

We will print the final scores and a few prediction rows. The row-count check confirms six windows, each with two products and seven dates.


In [ ]:
# Import the reader and exporter for saved analysis tables.
from autoforecast.results import export_results, read_workflow_results

# Show the four final-test scores with clear labels.
for metric, value in result.testing_metrics.items():
    print(metric, "=", value)

# Keep the preview small enough to show actual prediction values.
tables = read_workflow_results(result.path, include_tuning=True)
preview = tables.rows.to_pandas()[
    ["stage", "variant_id", "origin", "series_id", "target_date", "prediction"]
]
print(preview.head(10).to_string(index=False))
print("Forecast rows:", tables.rows.num_rows)
# Two trials on two earlier windows, plus validation and testing: six fit/window combinations.
assert tables.rows.num_rows == 84

# Check both the product population and the number of scored forecast cells.
summary = result.summary()
print(
    "Products: original =",
    summary["original_series"],
    "retained =",
    summary["retained_series"],
    "excluded =",
    summary["excluded_series"],
)
for stage, coverage in summary["coverage"].items():
    print(stage, coverage)

### Understand the output

You should see a readable prediction table and `Forecast rows: 84`. These are predictions, not input observations: 2 trials x 2 earlier windows, plus validation and test, each with 2 products x 7 dates.

In each row, `target_date` is the date predicted, `origin` is the last training date, and `prediction` is the predicted number of units. `series_id` is AutoForecast's internal product identifier. `variant_id` distinguishes the tuning trials; it is empty for final stages. For example, a row for product A on 15 January describes one prediction made using history through 14 January.

For the unchanged example, final-test MAE is about 2.83 units per product/day. RMSE is about 3.40 and gives larger errors more weight. WAPE is about 0.404, or 40.4% of total actual units. Bias is about +2.56: predictions are too high by 2.56 units on average. Negative bias means predictions are too low. Changed data or settings can produce different scores.

The product counts should be 2 original, 2 retained and 0 excluded. Each final stage should show `expected: 14` and `scored: 14`: every product/date cell was scored. `originally_missing`, `zero_filled` and `excluded` should all be zero. These are forecast-cell counts, distinct from the product counts above. Check them before comparing errors.


In file analysis, tuning rows have stage `selection` and a trial/attempt run ID. Published tables use stage `tuning`, the parent workflow run ID, and separate trial/window columns. A date may occur in several windows; keep the window identity when comparing forecasts.


## Step 4: Export the analysis tables

We will write an export to a new folder. The unique suffix avoids overwriting an earlier export.


In [ ]:
# Import the tools used in this step.
import uuid

# Use a new export folder so earlier files stay available.
export_path = work / ("analysis-" + uuid.uuid4().hex[:8])
export_results(tables, export_path)
print(export_path)

### Understand the output

You should see the new export folder path. Open it to inspect the analysis files. Exporting copies results for analysis; it does not retrain the model or change the retained forecasts.


Check the saved coverage and exclusion counts before comparing scores. Do not compare different populations as though they were the same experiment. Inspect `tables.runs` for per-window scores and artifact paths.
